In [1]:
import pandas as pd
import sys
from pathlib import Path

from sklearn.metrics import fbeta_score, make_scorer

# Works whether kernel cwd is project root or notebooks/
cwd = Path.cwd().resolve()
if (cwd / "src").exists():
    src_path = cwd / "src"
elif (cwd.parent / "src").exists():
    src_path = cwd.parent / "src"
else:
    raise RuntimeError("Could not find src/ directory")

sys.path.insert(0, str(src_path))

from exoplanet_detector.config import DEFAULT_RUN_TAG, get_run_artifact_dirs
from exoplanet_detector.features.feature_selection import FINAL_FEATURE_COLUMNS
from exoplanet_detector.models.feature_analysis import (
    build_evaluation_sets,
    build_feature_importance_matrix,
    compute_or_load_permutation_importance,
    get_feature_analysis_paths,
    load_deployed_models,
    predict_and_explain_single_row,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


In [2]:
RUN_TAG = DEFAULT_RUN_TAG
run_dirs = get_run_artifact_dirs(RUN_TAG, create=True)

DEPLOYMENT_ARTIFACT_DIR = run_dirs["deployment"]
DEPLOY_MODELS_PATH = DEPLOYMENT_ARTIFACT_DIR / "deploy_models.joblib"
DEPLOY_MANIFEST_PATH = DEPLOYMENT_ARTIFACT_DIR / "deploy_manifest.csv"

FEATURE_ANALYSIS_PATHS = get_feature_analysis_paths(RUN_TAG, create=True)
PERMUTATION_IMPORTANCE_PATH = FEATURE_ANALYSIS_PATHS["permutation_importance_path"]
FEATURE_ANALYSIS_META_PATH = FEATURE_ANALYSIS_PATHS["feature_analysis_meta_path"]
FEATURE_IMPORTANCE_MATRIX_PATH = FEATURE_ANALYSIS_PATHS["feature_importance_matrix_path"]

deployed_models, deployed_model_registry_df = load_deployed_models(DEPLOY_MODELS_PATH)
deploy_manifest = pd.read_csv(DEPLOY_MANIFEST_PATH)

print(f"Feature-analysis artifacts: {FEATURE_ANALYSIS_PATHS['artifact_dir']}")
deployed_model_registry_df

Feature-analysis artifacts: /Users/janma/Desktop/exoplanet-detection/artifacts/feature_analysis/v1


,deploy_id,model_name,profile,threshold
0,deploy_f2,knn,f2,0.363636
1,deploy_precision,hgb,precision_constrained,0.885432
2,deploy_recall,rf,recall_constrained,0.029039


In [3]:
KOI_test_set = pd.read_csv("../data/processed/KOI_test_set.csv")
K2P_set = pd.read_csv("../data/processed/K2P_set.csv")

In [4]:
evaluation_sets = build_evaluation_sets(
    KOI_test_set,
    K2P_set,
    feature_columns=FINAL_FEATURE_COLUMNS,
    include_combined=True,
)

In [5]:
FORCE_RECOMPUTE_FEATURE_ANALYSIS = False
N_REPEATS = 20
PERMUTATION_RANDOM_STATE = 42
PERMUTATION_N_JOBS = 1
PERMUTATION_SCORING_NAME = "f2"
PERMUTATION_SCORER = make_scorer(fbeta_score, beta=2, zero_division=0)

permutation_importance_df, feature_analysis_meta, loaded_from_cache = compute_or_load_permutation_importance(
    deployed_models,
    evaluation_sets,
    run_tag=RUN_TAG,
    permutation_importance_path=PERMUTATION_IMPORTANCE_PATH,
    feature_analysis_meta_path=FEATURE_ANALYSIS_META_PATH,
    scorer=PERMUTATION_SCORER,
    scoring_name=PERMUTATION_SCORING_NAME,
    n_repeats=N_REPEATS,
    random_state=PERMUTATION_RANDOM_STATE,
    n_jobs=PERMUTATION_N_JOBS,
    force_recompute=FORCE_RECOMPUTE_FEATURE_ANALYSIS,
)

if loaded_from_cache:
    print(f"Loaded cached permutation importance: {PERMUTATION_IMPORTANCE_PATH}")
else:
    print(f"Saved permutation importance: {PERMUTATION_IMPORTANCE_PATH}")
    print(f"Saved metadata: {FEATURE_ANALYSIS_META_PATH}")

permutation_importance_df

Loaded cached permutation importance: /Users/janma/Desktop/exoplanet-detection/artifacts/feature_analysis/v1/permutation_importance.csv


,deploy_id,model_name,profile,threshold,dataset,feature,importance_mean,importance_std,n_repeats,scoring,importance_rank
0,deploy_f2,knn,f2,0.363636,K2P,orbital_period_days,0.039938,0.007426,20,f2,1
1,deploy_f2,knn,f2,0.363636,K2P,transit_duration_hours,0.030641,0.004361,20,f2,2
2,deploy_f2,knn,f2,0.363636,K2P,transit_depth,0.017684,0.006941,20,f2,3
3,deploy_f2,knn,f2,0.363636,K2P,a_over_rs,0.016489,0.004517,20,f2,4
4,deploy_f2,knn,f2,0.363636,K2P,stellar_metallicity_dex,0.016095,0.004451,20,f2,5
...,...,...,...,...,...,...,...,...,...,...,...
94,deploy_recall,rf,recall_constrained,0.029039,KOI_test_plus_K2P,impact_parameter,0.014248,0.000994,20,f2,7
95,deploy_recall,rf,recall_constrained,0.029039,KOI_test_plus_K2P,stellar_logg_cgs,0.006482,0.000531,20,f2,8
96,deploy_recall,rf,recall_constrained,0.029039,KOI_test_plus_K2P,stellar_teff_k,0.006087,0.000643,20,f2,9
97,deploy_recall,rf,recall_constrained,0.029039,KOI_test_plus_K2P,transit_duration_hours,0.004770,0.000824,20,f2,10


In [6]:
feature_importance_matrix_df = build_feature_importance_matrix(
    permutation_importance_df,
    deploy_ids=list(deployed_models.keys()),
    dataset_names=list(evaluation_sets.keys()),
    feature_order=FINAL_FEATURE_COLUMNS,
)

feature_importance_matrix_df.to_csv(FEATURE_IMPORTANCE_MATRIX_PATH, index=False)
print(f"Saved feature importance matrix: {FEATURE_IMPORTANCE_MATRIX_PATH}")
print(f"Matrix shape: {feature_importance_matrix_df.shape}")
feature_importance_matrix_df

Saved feature importance matrix: /Users/janma/Desktop/exoplanet-detection/artifacts/feature_analysis/v1/feature_importance_matrix.csv
Matrix shape: (11, 10)


model_dataset,feature,deploy_f2__KOI_test,deploy_f2__K2P,deploy_f2__KOI_test_plus_K2P,deploy_recall__KOI_test,deploy_recall__K2P,deploy_recall__KOI_test_plus_K2P,deploy_precision__KOI_test,deploy_precision__K2P,deploy_precision__KOI_test_plus_K2P
0,orbital_period_days,0.074617,0.039938,0.072628,0.035122,0.000195,0.019047,0.103809,0.030832,0.071124
1,impact_parameter,0.019767,0.005691,0.001050,0.032403,-0.000440,0.014248,0.040011,0.008760,0.025768
2,transit_duration_hours,0.058593,0.030641,0.062038,0.009296,-0.000088,0.004770,0.084444,0.017087,0.046779
3,transit_depth,0.053896,0.017684,0.031936,0.062791,-0.002150,0.034220,0.138935,0.055926,0.086622
4,inclination_deg,0.022984,-0.000823,0.007216,0.050961,0.000442,0.027773,0.046802,0.002585,0.024084
5,a_over_rs,0.032001,0.016489,0.024727,0.048842,-0.000817,0.026464,0.210004,0.044286,0.131463
6,planet_radius_rearth,0.076522,0.002708,0.038951,0.068097,-0.000021,0.037464,0.165748,0.006537,0.084659
7,insolation_earth,0.030800,0.004623,0.019758,0.038444,-0.000036,0.018085,0.077360,0.001053,0.039635
8,stellar_teff_k,0.014325,-0.001042,0.009430,0.010156,-0.001233,0.006087,0.083105,0.035272,0.066172
9,stellar_logg_cgs,0.020333,-0.005813,0.005920,0.011963,-0.001130,0.006482,0.080990,0.051070,0.068631


The table displays each feature's permutation importance for each (model x dataset) pair. Some values are negative, which is not problematic.

In [9]:
sample_row_f2 = KOI_test_set.loc[:, FINAL_FEATURE_COLUMNS].iloc[[0]].copy()
background_f2 = evaluation_sets["KOI_test"][0]

f2_row_explanation = predict_and_explain_single_row(
    deployed_models,
    deploy_id="deploy_f2",
    row=sample_row_f2,
    background_data=background_f2,
    feature_columns=FINAL_FEATURE_COLUMNS,
    max_background_rows=200,
    random_state=42,
    kernel_nsamples=200,
    make_waterfall_plot=True,
    show_waterfall=False,
)

print(f"Deploy ID: {f2_row_explanation['deploy_id']}")
print(f"Model: {f2_row_explanation['model_name']} ({f2_row_explanation['profile']})")
print(f"Threshold: {f2_row_explanation['threshold']:.6f}")
print(f"Positive-class probability: {f2_row_explanation['probability_positive']:.6f}")
print(f"Negative-class probability: {f2_row_explanation['probability_negative']:.6f}")
print(f"Score (same as positive-class probability): {f2_row_explanation['score']:.6f}")
print(f"Prediction: {f2_row_explanation['prediction']}")
print(f"Explainer kind: {f2_row_explanation['explainer_kind']}")

f2_row_explanation["shap_values"].head(10)

Using 200 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.
100%|██████████| 1/1 [00:01<00:00,  1.46s/it]

Deploy ID: deploy_f2
Model: knn (f2)
Threshold: 0.363636
Positive-class probability: 0.000000
Negative-class probability: 1.000000
Score (same as positive-class probability): 0.000000
Prediction: 0
Explainer kind: kernel


planet_radius_rearth      -0.114116
stellar_metallicity_dex   -0.101233
inclination_deg           -0.081367
a_over_rs                 -0.038518
impact_parameter          -0.033368
transit_depth             -0.023107
transit_duration_hours     0.013853
stellar_teff_k            -0.012111
stellar_logg_cgs          -0.004275
insolation_earth          -0.002680
Name: shap_value, dtype: float64